# Synapse Analytics + Delta-RS Example (Spark-Free)

This notebook demonstrates **Spark-free Delta Lake operations** with **Azure Synapse Analytics** using LakeLogic and Delta-RS.

## 🚀 Features

- ✅ Read Synapse Analytics tables (no Spark!)
- ✅ Validate data with LakeLogic contracts
- ✅ Atomic MERGE operations (upsert)
- ✅ Inventory reconciliation & calculated fields
- ✅ Time travel & change tracking
- ✅ Azure AD authentication

## 📋 Prerequisites

```bash
pip install "lakelogic[delta]"
pip install azure-identity  # For Azure AD auth
```

## Setup: Configure Credentials

In [ ]:
import os
from lakelogic import DataProcessor
from lakelogic.engines.delta_adapter import DeltaAdapter
from lakelogic.engines.unity_catalog import resolve_catalog_path
import polars as pl

# Required: Synapse storage account
os.environ["SYNAPSE_STORAGE_ACCOUNT"] = "mysynapsestorage"

# Option 1: Account Key Authentication
os.environ["AZURE_STORAGE_ACCOUNT_NAME"] = "mysynapsestorage"
os.environ["AZURE_STORAGE_ACCOUNT_KEY"] = "..."

# Option 2: Azure AD Authentication (Recommended)
# Run: az login
# Credentials are automatically picked up

print("✅ Credentials configured")

## Example 1: Read Synapse Analytics Table

Use Synapse table names directly (`database.schema.table`) - LakeLogic automatically resolves them to ADLS Gen2 paths!

In [ ]:
# Create processor with Polars engine (no Spark!)
processor = DataProcessor(
    engine="polars",
    contract="synapse_analytics_contract.yaml"
)

# Read Synapse Analytics table
good_df, bad_df = processor.run_source("inventorydb.dbo.stock_levels")

print(f"✅ Good records: {len(good_df)}")
print(f"❌ Quarantined records: {len(bad_df)}")

# Display good data
good_df.head()

In [ ]:
# Display quarantined data (if any)
if len(bad_df) > 0:
    print("Quarantined records:")
    bad_df.head()
else:
    print("No quarantined records! 🎉")

## Example 2: Table Name Resolution

See how Synapse table names are automatically resolved to ADLS Gen2 paths.

In [ ]:
# Resolve Synapse table name
table_name = "inventorydb.dbo.stock_levels"
storage_path = resolve_catalog_path(table_name, platform="synapse")

print(f"Table name: {table_name}")
print(f"ADLS Gen2 path: {storage_path}")
print(f"\n✅ LakeLogic automatically handles this resolution!")

## Example 3: MERGE Operation (Upsert)

Perform atomic MERGE operations **without Spark** using composite keys!

In [ ]:
# Create new/updated inventory data
new_inventory = pl.DataFrame({
    "product_id": ["PROD-001", "PROD-002", "PROD-999"],
    "warehouse_id": ["WH-01", "WH-01", "WH-02"],
    "quantity_on_hand": [100, 50, 200],
    "quantity_reserved": [10, 5, 20],
    "quantity_available": [90, 45, 180],
    "reorder_point": [50, 25, 100],
    "last_updated": ["2026-02-09T10:00:00Z", "2026-02-09T10:00:00Z", "2026-02-09T10:00:00Z"],
    "location": ["Aisle A1", "Aisle A2", "Aisle B1"]
})

print("New/updated inventory:")
new_inventory

In [ ]:
# MERGE into Synapse table (atomic, no Spark!)
adapter = DeltaAdapter()
stats = adapter.merge(
    target_path="inventorydb.dbo.stock_levels",
    source_df=new_inventory,
    merge_key=["product_id", "warehouse_id"]  # Composite key
)

print(f"✅ MERGE complete:")
print(f"  - Updated: {stats['num_updated']} records")
print(f"  - Inserted: {stats['num_inserted']} records")

## Example 4: Inventory Reconciliation

LakeLogic automatically calculates inventory metrics using contract transformations.

In [ ]:
# Create inventory data with missing calculated fields
inventory_data = pl.DataFrame({
    "product_id": ["PROD-100", "PROD-101"],
    "warehouse_id": ["WH-03", "WH-03"],
    "quantity_on_hand": [150, 75],
    "quantity_reserved": [30, 15],
    "quantity_available": [None, None],  # Will be calculated
    "reorder_point": [50, 25],
    "last_updated": ["2026-02-09T11:00:00Z", "2026-02-09T11:00:00Z"],
    "location": ["Aisle C1", "Aisle C2"]
})

print("Before processing:")
inventory_data

In [ ]:
# Process with contract (calculates quantity_available and needs_reorder)
processor = DataProcessor(
    engine="polars",
    contract="synapse_analytics_contract.yaml"
)
good_df, bad_df = processor.run(inventory_data)

print("After processing:")
good_df[["product_id", "quantity_on_hand", "quantity_reserved", "quantity_available", "needs_reorder"]]

## Example 5: Reorder Report

Identify products that need reordering.

In [ ]:
# Read inventory data
df = adapter.read("inventorydb.dbo.stock_levels")

# Filter products that need reordering
needs_reorder = df.filter(pl.col("quantity_available") <= pl.col("reorder_point"))

print(f"⚠️ Products needing reorder: {len(needs_reorder)}")
needs_reorder[["product_id", "warehouse_id", "quantity_available", "reorder_point", "location"]]

In [ ]:
# Calculate reorder quantities (reorder to 2x reorder point)
reorder_report = needs_reorder.with_columns([
    (pl.col("reorder_point") * 2 - pl.col("quantity_available")).alias("reorder_quantity")
])

print("Reorder Report:")
reorder_report[["product_id", "warehouse_id", "quantity_available", "reorder_quantity"]]

## Example 6: Inventory Analytics

Analyze inventory levels across warehouses.

In [ ]:
# Inventory summary by warehouse
warehouse_summary = df.group_by("warehouse_id").agg([
    pl.count().alias("product_count"),
    pl.sum("quantity_on_hand").alias("total_on_hand"),
    pl.sum("quantity_reserved").alias("total_reserved"),
    pl.sum("quantity_available").alias("total_available")
]).sort("total_available", descending=True)

print("Inventory Summary by Warehouse:")
warehouse_summary

In [ ]:
# Products with low stock (< 20% of reorder point)
low_stock = df.filter(
    pl.col("quantity_available") < (pl.col("reorder_point") * 0.2)
)

print(f"🚨 Critical low stock: {len(low_stock)} products")
low_stock[["product_id", "warehouse_id", "quantity_available", "reorder_point"]]

## Example 7: Time Travel & Change Tracking

Track inventory changes over time.

In [ ]:
# Read current inventory
df_current = adapter.read("inventorydb.dbo.stock_levels")
print(f"Current: {len(df_current)} records")

# Read yesterday's inventory
df_yesterday = adapter.read(
    "inventorydb.dbo.stock_levels",
    timestamp="2026-02-08T00:00:00Z"
)
print(f"Yesterday: {len(df_yesterday)} records")

In [ ]:
# Compare inventory changes
if len(df_yesterday) > 0 and len(df_current) > 0:
    changes = df_current.join(
        df_yesterday,
        on=["product_id", "warehouse_id"],
        suffix="_yesterday"
    ).with_columns([
        (pl.col("quantity_on_hand") - pl.col("quantity_on_hand_yesterday")).alias("quantity_change")
    ])
    
    print("Inventory Changes (Top 10):")
    changes.sort("quantity_change", descending=True).head(10)[
        ["product_id", "warehouse_id", "quantity_on_hand", "quantity_on_hand_yesterday", "quantity_change"]
    ]

In [ ]:
# Get table history
history = adapter.get_history("inventorydb.dbo.stock_levels", limit=10)
print("Table history (last 10 commits):")
history

## Example 8: Complete Pipeline (Full Refresh)

Read → Validate → Overwrite for inventory snapshots.

In [ ]:
# Step 1: Read from Synapse
print("Step 1: Reading from Synapse...")
processor = DataProcessor(
    engine="polars",
    contract="synapse_analytics_contract.yaml"
)
good_df, bad_df = processor.run_source("inventorydb.dbo.stock_levels")
print(f"  ✅ Good: {len(good_df)}, ❌ Bad: {len(bad_df)}")

# Step 2: Overwrite validated table (full refresh)
print("\nStep 2: Overwriting validated table...")
adapter = DeltaAdapter()
adapter.write(
    df=good_df,
    path="inventorydb.dbo.stock_levels_validated",
    mode="overwrite"  # Full refresh for inventory
)
print(f"  ✅ Wrote {len(good_df)} validated records")

# Step 3: Write quarantined data
if len(bad_df) > 0:
    print("\nStep 3: Writing quarantined data...")
    adapter.write(
        df=bad_df,
        path="inventorydb.dbo.stock_levels_quarantine",
        mode="append"
    )
    print(f"  ✅ Wrote {len(bad_df)} quarantined records")

print("\n✅ Pipeline complete!")

## Example 9: Azure AD Authentication (Recommended)

Use Azure AD for more secure authentication.

In [ ]:
from azure.identity import DefaultAzureCredential

# Get Azure AD token
credential = DefaultAzureCredential()
token = credential.get_token("https://storage.azure.com/.default")

# Create adapter with Azure AD auth
adapter_ad = DeltaAdapter(storage_options={
    "AZURE_STORAGE_ACCOUNT_NAME": "mysynapsestorage",
    "BEARER_TOKEN": token.token
})

# Read with Azure AD auth
df = adapter_ad.read("inventorydb.dbo.stock_levels")
print(f"✅ Read {len(df)} records using Azure AD authentication")

## 🎯 Summary

### What We Demonstrated:

✅ **Synapse Analytics table names** - Use `database.schema.table` directly  
✅ **Spark-free Delta Lake** - Read/write with Polars (10-100x faster)  
✅ **Atomic MERGE** - Upsert with composite keys  
✅ **Inventory reconciliation** - Automatic calculated fields  
✅ **Reorder reports** - Business intelligence queries  
✅ **Time travel & change tracking** - Historical analysis  
✅ **Full refresh mode** - Inventory snapshots  
✅ **Azure AD auth** - Secure authentication  

### Learn More:

- 📚 [Delta Lake Support](../../docs/delta_lake_support.md)
- 📚 [Catalog Table Names](../../docs/catalog_table_names.md)
- 📚 [Synapse Analytics Contract](synapse_analytics_contract.yaml)

---

*Last Updated: February 2026*